# Section 2.3.3: DeepHit (Single Event) Demo

This demo covers how to implement DeepHit (Lee et al., 2018) specialized to working with a single critical event (as in Example 2.3.1 of the monograph) with the help of PyCox and PyTorch. This demo largely builds off the exponential time-to-event prediction model demo (`S2.2.2_Exponential.ipynb`), so please look at that first as it describes the different steps of the experimental setup in more detail.

In this demo, we specifically use PyCox's `DeepHitSingle` model (which is a Python class). Here, the word "single" shows up since we are using DeepHit with a single critical event of interest (in general, DeepHit handles competing risks, for which there is more than just a single critical event of interest).

As discussed in the monograph, DeepHit is inherently a discrete time model that requires that we specify what time grid to use, so when we set up the data to be used with PyTorch training, there's an extra step of setting up a time grid and discretizing the observed times seen in the training and validation data.

*Disclaimer:* Note that PyCox actually uses various conventions that are not standard in terms of working with PyTorch (it relies heavily on another package `torchtuples` by the same author as PyCox). Our demo here intentionally tries to stick to what would be considered standard PyTorch conventions (so that as a warning up front, what we present here does *not* perfectly align with the demos that come with PyCox but will more closely resemble how PyTorch is commonly used in many other settings, including for instance how model training works in TorchSurv). The key reason why we stick to standard PyTorch conventions is so that the code makes various preprocessing and model training steps more transparent, so that if you want to modify any part of these, it should be easier to do so (for instance, if you want to introduce a specific learning rate schedule, if you want to add early stopping based on a validation set integrated Brier score, if you want to control for randomness in a particular way using your own data loaders or specific random number generators, etc).

### Loading in the SUPPORT dataset (Knaus et al., 1995)

As with the earlier demo, we begin by defining the training, validation, and test sets.

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

X_full_train_raw_np, Y_full_train_np, D_full_train_np, \
        X_test_raw_np, Y_test_np, D_test_np, \
        features_before_preprocessing, features_after_preprocessing, \
        events, train_test_split_prespecified, \
        build_preprocessor_and_preprocess, apply_preprocessor = load_dataset('support')

# split the "full training set" into the actual training set and a validation set (using a 80/20 split)
X_train_raw_np, X_val_raw_np, Y_train_np, Y_val_np, D_train_np, D_val_np = \
    train_test_split(X_full_train_raw_np, Y_full_train_np, D_full_train_np,
                     test_size=.2, random_state=0)

print(f'Training set size {X_train_raw_np.shape[0]}')
print(f'Validation set size {X_val_raw_np.shape[0]}')
print(f'Test set size {X_test_raw_np.shape[0]}')
print()

print(f'Features before preprocessing ({len(features_before_preprocessing)} total):')
print(features_before_preprocessing)
print()

print(f'Features after preprocessing ({len(features_after_preprocessing)} total):')
print(features_after_preprocessing)
print()

print('Events:', events)  # only one critical event for the standard time-to-event prediction setup
print()

We fit and apply a preprocessor to the training set. We apply (but do not re-fit) the preprocessor to the validation set.

In [ ]:
X_train_np, preprocessor = build_preprocessor_and_preprocess(X_train_raw_np)
X_val_np = apply_preprocessor(X_val_raw_np, preprocessor)

### Discretizing time

We use PyCox's provided `LabTransDiscreteTime` class that helps discretize time. Note that there are many ways to discretize time (see Section 2.3.2 of the monograph). Here, we are only implementing two ways (using all unique times of death, or discretizing based on some user-specified number of quantiles).

In [ ]:
import numpy as np
from pycox.preprocessing.label_transforms import LabTransDiscreteTime

num_time_steps = 128  # set this to 0 to use all unique times of death

if num_time_steps == 0:
    mask = (D_train_np == 1)  # boolean mask specifying which training patients experienced death
    label_transform = LabTransDiscreteTime(np.unique(Y_train_np[mask]))
else:
    # use a quantile based discretization, which could possibly end up using fewer than the
    # number of time steps requested (if it turns out that in the dataset, there are many
    # duplicate observed times)
    label_transform = LabTransDiscreteTime(num_time_steps, scheme='quantiles')

Y_train_discrete_np, D_train_discrete_np = label_transform.fit_transform(Y_train_np, D_train_np)
Y_val_discrete_np, D_val_discrete_np = label_transform.transform(Y_val_np, D_val_np)
# note: PyCox's discretization code allows for the possibility that the event indicator changes
# (if a patient's survival time is known so that the event indicator is 1, but the survival time
# is after the maximum discrete time grid time, then the patient's event indicator is changed to
# being censored)

time_grid_train_np = label_transform.cuts
output_num_time_steps = len(time_grid_train_np)
print(f'Number of time steps to be used with DeepHit: {output_num_time_steps}')
print('Time grid:', time_grid_train_np)

### Preparing training and validation sets for use with PyTorch

Note that at this point the observed times have been converted into discrete indices. Note that at the time of writing, the code is a bit picky about the data types used.

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_train = torch.tensor(X_train_np, dtype=torch.float32, device=device)
Y_train = torch.tensor(Y_train_discrete_np, dtype=torch.int64, device=device)
D_train = torch.tensor(D_train_discrete_np, dtype=torch.int32, device=device)
train_data = list(zip(X_train, Y_train, D_train))

X_val = torch.tensor(X_val_np, dtype=torch.float32, device=device)
Y_val = torch.tensor(Y_val_discrete_np, dtype=torch.int64, device=device)
D_val = torch.tensor(D_val_discrete_np, dtype=torch.int32, device=device)
val_data = list(zip(X_val, Y_val, D_val))

### Setting up the DeepHit model and loss

We are now ready to set up the DeepHit model using PyCox's `DeepHitSingle` class. Note that PyCox's implementation of DeepHit does *not* require that what we call `base_neural_net` outputs valid probability distributions. The reason is that under the hood, when it computes the DeepHit loss, it actually applies a softmax activation right after the base neural net (specifically, it applies equation (9) of Kvamme and Borgan (2021)).

As we shall see later in the demo, the `DeepHitSingle` class provides various helper functions such as predicting survival functions for test data and interpolating these survival functions so that they use a higher time resolution than what we specified in `time_grid_train_np`.

In [ ]:
import torch.nn as nn
from pycox.models import DeepHitSingle

torch.manual_seed(0)

num_input_features = X_train.size(1)

base_neural_net = nn.Sequential(nn.Linear(num_input_features, 8),
                                nn.ReLU(),
                                nn.Linear(8, output_num_time_steps)).to(device)  # no softmax needed at the end

# setting alpha=1.0 means that we only use the negative log likelihood loss and not also the ranking loss
# (we do this just as an illustrative example as it corresponds to how DeepHit is first presented in
# Example 2.3.1 of the monograph)
deephit_model = DeepHitSingle(base_neural_net, alpha=1.0, device=device,
                              duration_index=time_grid_train_np)
deephit_loss = deephit_model.loss  # we'll see how to use this loss function in the next code cell

### Training the model using minibatch gradient descent

Just as in the exponential time-to-event prediction model demo, we use the validation set to decide on whether to use a previous epoch's learned model parameters.

In [ ]:
from copy import deepcopy
from torch.utils.data import DataLoader
from pycox.models.data import pair_rank_mat

# minibatch gradient descent
num_epochs = 100
batch_size = 128
learning_rate = 1e-3

train_loader = DataLoader(train_data, batch_size, shuffle=True)  # shuffling for minibatch gradient descent
val_loader = DataLoader(val_data, batch_size, shuffle=False)  # there is no need to shuffle the validation data

optimizer = torch.optim.Adam(base_neural_net.parameters(), lr=learning_rate)
train_epoch_losses = []
val_epoch_losses = []
best_val_loss = float('inf')
best_params = None
best_epoch_index = None
for epoch_index in range(num_epochs):
    base_neural_net.train()
    for X_batch, Y_batch, D_batch in train_loader:
        neural_net_output = base_neural_net(X_batch)

        # note that the loss function requires a ranking matrix to be specified;
        # this is specific to the ranking loss that we are actually not using in this
        # particular demo (but we leave this here since one could modify the demo
        # easily to also be using the ranking loss, in which case this ranking matrix
        # will be needed)
        rank_mat = pair_rank_mat(Y_batch.cpu().numpy(), D_batch.cpu().numpy())
        rank_mat = torch.tensor(rank_mat, dtype=torch.int, device=device)

        loss_batch = deephit_loss(neural_net_output, Y_batch, D_batch, rank_mat)

        optimizer.zero_grad()
        loss_batch.backward()
        optimizer.step()

    # evaluate training and validation set losses
    # (note that in practice, instead of evaluating the negative log likelihood loss,
    # we could instead evaluate other metrics such as time-dependent concordance index,
    # integrated Brier score, etc)
    base_neural_net.eval()
    with torch.no_grad():
        train_loss = torch.tensor(0.0, dtype=torch.float, device=device)
        num_points = 0
        for X_batch, Y_batch, D_batch in train_loader:
            batch_num_points = X_batch.size(0)
            neural_net_output = base_neural_net(X_batch)
            rank_mat = pair_rank_mat(Y_batch.cpu().numpy(), D_batch.cpu().numpy())
            rank_mat = torch.tensor(rank_mat, dtype=torch.int, device=device)
            train_loss += deephit_loss(neural_net_output, Y_batch, D_batch, rank_mat) * batch_num_points
            num_points += batch_num_points
        train_loss = float(train_loss / num_points)
        train_epoch_losses.append(train_loss)
        print(f'Epoch {epoch_index + 1} - train loss {train_loss}', end=' ', flush=True)

        val_loss = torch.tensor(0.0, dtype=torch.float, device=device)
        num_points = 0
        for X_batch, Y_batch, D_batch in val_loader:
            batch_num_points = X_batch.size(0)
            neural_net_output = base_neural_net(X_batch)
            rank_mat = pair_rank_mat(Y_batch.cpu().numpy(), D_batch.cpu().numpy())
            rank_mat = torch.tensor(rank_mat, dtype=torch.int, device=device)
            val_loss += deephit_loss(neural_net_output, Y_batch, D_batch, rank_mat) * batch_num_points
            num_points += batch_num_points
        val_loss = float(val_loss / num_points)
        val_epoch_losses.append(val_loss)
        print(f'- val loss {val_loss}', flush=True)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch_index = epoch_index
            best_params = deepcopy(base_neural_net.state_dict())
print(f'Best validation loss ({best_val_loss}) achieved at epoch {best_epoch_index + 1}')
base_neural_net.load_state_dict(best_params)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.plot(range(1, num_epochs + 1), train_epoch_losses, label='Training')
plt.plot(range(1, num_epochs + 1), val_epoch_losses, '--', label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

### Predicting test set survival curves

We begin by preprocessing the test data using the preprocessor built using training data.

In [ ]:
X_test_np = apply_preprocessor(X_test_raw_np, preprocessor).astype('float32')

PyCox has built-in functionality for predicting survival functions:

In [ ]:
surv_test_np = deephit_model.predict_surv(X_test_np, batch_size=batch_size, to_cpu=True, numpy=True)
surv_test_np.shape

Note that the time grid used is precisely the one stored in `time_grid_train_np`:

In [ ]:
len(time_grid_train_np)

PyCox provides a helper function `predict_surv_df` that gets both the time grid and the predicted survival functions into a Pandas DataFrame. Note that for this DataFrame, the rows correspond to time and columns correspond to the different test points.

In [ ]:
surv_test_df = deephit_model.predict_surv_df(X_test_np, batch_size=batch_size)
surv_test_df

If you want to interpolate, PyCox provides the following interpolation functionality:

In [ ]:
# here we interpolate by a factor of 10 (so that if we initially had m time steps, we would now have (m-1)*10 + 1 time steps)
interpolation_factor = 10
surv_test_interp_df = deephit_model.interpolate(interpolation_factor).predict_surv_df(X_test_np, batch_size=batch_size)
surv_test_interp_df

In [ ]:
time_grid_train_interp_np = surv_test_interp_df.index.to_numpy()  # the interpolated time grid is an interpolated version of the *training*  time grid
surv_test_interp_np = surv_test_interp_df.to_numpy().T

We point out that if one did not want to create a Pandas DataFrame first, then the following code could be used instead to produce the same interpolated time grid and interpolated survival function:

```
surv_test_interp_np = deephit_model.interpolate(interpolation_factor).predict_surv(X_test_np, batch_size=batch_size, to_cpu=True, numpy=True)
from pycox.models.utils import make_subgrid
time_grid_train_interp_np = np.array(make_subgrid(time_grid_train_np, interpolation_factor))
```

Here's a plot of the predicted survival function for the 0th test data point:

In [ ]:
plt.plot(time_grid_train_interp_np, surv_test_interp_np[0])
plt.xlabel('Time')
plt.ylabel('Survival probability')

### Computing test set evaluation metrics

Here, we use the interpolated survival functions (along with their corresponding interpolated time grid).

As for choosing evaluation times, we still pick these based on the test set's observed times.

In [ ]:
time_grid_test_np = np.unique(Y_test_np)

Since DeepHit does not make the proportional hazards assumption and can predict survival curves without shape constraints, Harrell's concordance index (Harrell et al., 1982) is not a valid evaluation metric to use.

#### Time-dependent concordance index (Antolini et al., 2005) using the PyCox implementation (Kvamme et al., 2019)

In [ ]:
from pycox.evaluation import EvalSurv

eval_pycox = EvalSurv(surv_test_interp_df, Y_test_np, D_test_np)
C_td = eval_pycox.concordance_td('antolini')
print(f'Time-dependent concordance index: {C_td}')

#### Truncated time-dependent concordance index (Uno et al., 2011) using the scikit-survival implementation (Pölsterl, 2020)

In [ ]:
from sksurv.util import Surv
from sksurv.metrics import concordance_index_ipcw

# convert training and test labels into the structured array format used by scikit-survival
labels_train_sksurv = Surv.from_arrays(D_train_np, Y_train_np)
labels_test_sksurv = Surv.from_arrays(D_test_np, Y_test_np)

# as an illustrative example, we evaluate the truncated time-dependent concordance index at
# 25, 50, and 75 percentile values of the time grid we specified earlier
eval_time_indices = [int(.25 * len(time_grid_test_np)),
                     int(.5 * len(time_grid_test_np)),
                     int(.75 * len(time_grid_test_np))]

for eval_time_index in eval_time_indices:
    eval_time = time_grid_test_np[eval_time_index]

    # find the interpolated time grid's time point closest to the evaluation time
    interp_time_index = np.argmin(np.abs(eval_time - time_grid_train_interp_np))
    surv_values_at_eval_time_np = surv_test_interp_np[:, interp_time_index]

    estimated_risks_np = 1 - surv_values_at_eval_time_np
    concordance = concordance_index_ipcw(labels_train_sksurv, labels_test_sksurv, estimated_risks_np, tau=eval_time)[0]
    print(f'Eval time {eval_time} - truncated time-dependent concordance: {concordance}')

#### Time-dependent AUC (Uno et al., 2007; Hung and Chiang, 2010) using the scikit-survival implementation (Pölsterl, 2020)

In [ ]:
from sksurv.metrics import cumulative_dynamic_auc

for eval_time_index in eval_time_indices:
    eval_time = time_grid_test_np[eval_time_index]

    # find the interpolated time grid's time point closest to the evaluation time
    interp_time_index = np.argmin(np.abs(eval_time - time_grid_train_interp_np))
    surv_values_at_eval_time_np = surv_test_interp_np[:, interp_time_index]

    estimated_risks_np = 1 - surv_values_at_eval_time_np
    AUC = cumulative_dynamic_auc(labels_train_sksurv, labels_test_sksurv, estimated_risks_np, times=[eval_time])[0][0]
    print(f'Eval time {eval_time} - time-dependent AUC: {AUC}')

#### Brier score (Graf et al., 1999) using the SurvivalEVAL implementation (Qi et al., 2024)

In [ ]:
from SurvivalEVAL.Evaluator import SurvivalEvaluator

eval = SurvivalEvaluator(surv_test_interp_np, time_grid_train_interp_np,
                         Y_test_np, D_test_np,
                         Y_train_np, D_train_np)  # note: training labels are also needed

for eval_time_index in eval_time_indices:
    eval_time = float(time_grid_test_np[eval_time_index])
    print(f'Eval time {eval_time} - Brier score: {eval.brier_score(eval_time)}')

IBS = eval.integrated_brier_score()
print(f'Integrated Brier score: {IBS}')

#### D-Calibration (Haider et al., 2020) using the SurvivalEVAL implementation (Qi et al., 2024)

In [ ]:
p_value, bin_stats = eval.d_calibration()
if p_value >= .05:
    print('The model is D-calibrated.')
else:
    print('The model is not D-calibrated.')

#### Mean Absolute Error via the margin (Haider et al., 2020) and, separately, the Pseudo Observation approach (Qi et al., 2023) both using the SurvivalEVAL implementation (Qi et al., 2024)

In [ ]:
weighted_MAE_margin = eval.mae(method='Margin', weighted=True)
print(f'Weighted MAE-Margin: {weighted_MAE_margin}')

In [ ]:
weighted_MAE_PO = eval.mae(method='Pseudo_obs', weighted=True)
print(f'Weighted MAE-PO: {weighted_MAE_PO}')